In [5]:
import sys, os; sys.path.append("..")
os.makedirs("../data/img2", exist_ok=True)
import requests
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D
import contextily as ctx
from pyproj import Transformer
from src.config import ESTACIONES, UTM_ESTACIONES
from src.meteo import utm_a_latlon


In [9]:
SUBZONAS = {
    "Coronel-Lota": (["831", "878", "880", "881", "879", "859", "860"], "#eb6834"),
    "Talcahuano-Hualpen": (["805", "838", "804", "802", "807", "806"], "#2a78d6"),
    "Concepcion": (["827"], "#e87ba4"),
    "Chiguayante": (["854"], "#eda100"),
    "Hualqui": (["841"], "#0bde75"),
}
COMUNA_SZ = {"Coronel": "Coronel-Lota", "Lota": "Coronel-Lota",
             "Talcahuano": "Talcahuano-Hualpen", "Hualpén": "Talcahuano-Hualpen",
             "Concepción": "Concepcion", "Chiguayante": "Chiguayante", "Hualqui": "Hualqui"}

color_de = {c: col for cods, col in SUBZONAS.values() for c in cods}
sz_de = {c: sz for sz, (cods, _) in SUBZONAS.items() for c in cods}

a_merc = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)  # coordenadas del mapa web
coords = utm_a_latlon(UTM_ESTACIONES)
xy = {c: a_merc.transform(lon, lat) for c, (lat, lon) in coords.items()}

geo = requests.get("https://raw.githubusercontent.com/caracena/chile-geojson/master/8.geojson", timeout=60).json()
comunas = [f for f in geo["features"] if f["properties"]["Comuna"] in COMUNA_SZ]

def anillos(geom):
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    for p in polys:
        yield [a_merc.transform(lon, lat) for lon, lat in p[0]]

FONDO = ctx.providers.Esri.WorldGrayCanvas      # gratuito, sin clave
X0, Y0 = a_merc.transform(-73.24, -37.13)
X1, Y1 = a_merc.transform(-72.88, -36.68)

def preparar(ax, relleno=None):
    for f in comunas:
        nom = f["properties"]["Comuna"]
        for ring in anillos(f["geometry"]):
            ax.add_patch(Polygon(ring, closed=True, ec="#444", lw=0.8,
                                 fc=relleno(nom) if relleno else "none", alpha=0.25 if relleno else 1))
    ax.set_xlim(X0, X1); ax.set_ylim(Y0, Y1)
    ctx.add_basemap(ax, source=FONDO, zoom=12)
    ax.set_axis_off()

# ---------- 2. SUB-ZONAS ----------
fig, ax = plt.subplots(figsize=(8, 10))
preparar(ax, relleno=lambda nom: SUBZONAS[COMUNA_SZ[nom]][1])
for c, (x, y) in xy.items():
    ax.scatter(x, y, s=70, c=color_de[c], edgecolors="white", linewidths=1.5, zorder=5)
ax.legend(handles=[Line2D([], [], marker="o", ls="", ms=9, mfc=col, mec="white", label=f"{sz} ({len(cods)} estaciones)")
                   for sz, (cods, col) in SUBZONAS.items()], loc="lower right", fontsize=9)
ax.set_title("Nivel 2 · Sub-zonas (un modelo por sub-zona)", loc="left", fontsize=12, fontweight="bold")
fig.savefig("../data/img2/mapa_2_subzonas.png", dpi=200, bbox_inches="tight"); plt.close(fig)
